In [ ]:
# !git pull

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 1 · [BOOTSTRAP]  Auto-setup: Local Windows + IIT GN + Google Cloud  ║
# ║  Run this cell FIRST every session — clones/pulls repo & installs deps.   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import os, sys, subprocess, platform
from pathlib import Path

GITHUB_REPO   = 'https://github.com/DhruvalPtl/alpha-synuclein-agent.git'
CLOUD_INSTALL = Path.home() / 'agent_workspace'
LOCAL_WIN     = Path(r'd:\3rd sem M.tech\agent_workspace')
LOCAL_IIT     = Path('/home/dhruval/agent_workspace')
LOCAL_WSL     = Path('/mnt/d/3rd sem M.tech/agent_workspace')

def _on_cloud():
    try:
        import urllib.request
        req = urllib.request.Request(
            'http://metadata.google.internal/',
            headers={'Metadata-Flavor': 'Google'}
        )
        urllib.request.urlopen(req, timeout=0.8)
        return True
    except Exception:
        pass
    # NOT on cloud if any local path exists
    return not (LOCAL_WIN.exists() or LOCAL_WSL.exists() or LOCAL_IIT.exists())

# ── Detect runtime environment ────────────────────────────────────────────────
# RUNTIME = 'cloud' -> Google Cloud / GCE VM
# RUNTIME = 'iit'   -> IIT GN Linux server (NVIDIA GPU present, no LOCAL_WIN)
# RUNTIME = 'local' -> local Windows machine (LOCAL_WIN exists)
if LOCAL_WIN.exists():
    RUNTIME = 'local'
elif LOCAL_IIT.exists():
    RUNTIME = 'iit'
elif _on_cloud():
    RUNTIME = 'cloud'
else:
    RUNTIME = 'local'  # fallback

IS_CLOUD = (RUNTIME == 'cloud')   # backward-compat alias kept for other cells

print(f'Environment : {RUNTIME.upper()}')
print(f'Platform    : {platform.system()} {platform.machine()}')
print(f'Python      : {sys.version.split()[0]}  ({sys.executable})')

# ── Set PROJECT_ROOT based on runtime ────────────────────────────────────────
if RUNTIME == 'cloud':
    PROJECT_ROOT = CLOUD_INSTALL
elif RUNTIME == 'iit':
    PROJECT_ROOT = LOCAL_IIT
else:  # local
    PROJECT_ROOT = LOCAL_WSL if LOCAL_WSL.exists() else LOCAL_WIN

if RUNTIME in ('cloud', 'iit'):
    # ── Always check .git, not just directory existence ──────────────────────
    if not (PROJECT_ROOT / '.git').exists():
        print(f'\nNo git repo at {PROJECT_ROOT} — cloning ...')
        if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
            print('  (directory exists but is not a git repo — removing it first)')
            import shutil
            shutil.rmtree(PROJECT_ROOT)
        r = subprocess.run(
            ['git', 'clone', GITHUB_REPO, str(PROJECT_ROOT)],
            text=True, capture_output=True
        )
        if r.returncode != 0:
            raise RuntimeError(
                f'git clone failed (exit {r.returncode}):\n{r.stderr}\n'
                'Check your internet connection on the instance.'
            )
        print('Clone complete.')
    else:
        print(f'\nRepo found at {PROJECT_ROOT} — pulling latest ...')
        r = subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull'],
            text=True, capture_output=True
        )
        out = r.stdout.strip() or r.stderr.strip()
        print(out)
        if r.returncode != 0:
            print('WARNING: git pull failed — continuing with existing local copy.')
        print('Rebuilding leaderboard from disk after pull ...')
        rb_r = subprocess.run(
            [sys.executable, '-m', 'agent.tools.rebuild_leaderboard'],
            capture_output=True, text=True, cwd=str(PROJECT_ROOT)
        )
        print('Leaderboard rebuilt.' if rb_r.returncode == 0
              else f'Leaderboard rebuild skipped: {rb_r.stderr[:100]}')
else:
    print(f'\nLocal workspace: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Working dir : {os.getcwd()}')

def _has_nvidia():
    try:
        return subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

if RUNTIME == 'cloud' and _has_nvidia():
    print('\nGPU detected — installing PyTorch CUDA 12.1 ...')
    r = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ], capture_output=True, text=True)
    print('PyTorch CUDA OK.' if r.returncode == 0
          else f'CUDA wheel failed, CPU fallback: {r.stderr[:200]}')
elif RUNTIME == 'cloud':
    print('\nNo GPU — CPU PyTorch will install via requirements.')

print('\nInstalling requirements.txt ...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--break-system-packages', '-r',
     str(PROJECT_ROOT / 'requirements.txt')],
    text=True, capture_output=True
)
if r.returncode != 0:
    print('[STDERR]', r.stderr[-2000:])
    raise RuntimeError('pip install failed — see above')
print('Requirements OK.')

# ── Install package in editable mode ──────────────────────────────────────────
# This makes 'import agent' work from ANY cell even after kernel restart.
print('\nInstalling package (editable) so import agent works everywhere ...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--break-system-packages','-e', str(PROJECT_ROOT)],
    text=True, capture_output=True, cwd=str(PROJECT_ROOT)
)
if r.returncode == 0:
    print('Package installed — import agent now works from any cell.')
else:
    print(f'Editable install skipped (fallback to sys.path): {r.stderr[:100]}')

for d in ['master_log', 'data/raw', 'data/splits', 'data/processed', 'experiments', 'sessions']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

env_path = PROJECT_ROOT / '.env'
if not env_path.exists():
    env_path.write_text(
        '# Fill in the API keys you want to use\n'
        'GEMINI_API_KEY=\nGROQ_API_KEY=\nMISTRAL_API_KEY=\n'
        'OPENROUTER_API_KEY=\nCEREBRAS_API_KEY=\n'
        '# Machine tag (avoids experiment ID collisions across machines)\n'
        'MACHINE_ID=iit\n'
    )
    print(f'\nCreated .env at {env_path}  <-- edit MACHINE_ID and API keys!')
else:
    print(f'\n.env found at {env_path}')

if _has_nvidia():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'\nGPU : {r.stdout.strip()}')

import os as _os
machine_id = _os.environ.get('MACHINE_ID', '') or platform.node()
print(f'\nMACHINE_ID  : {machine_id}  (experiments tagged with this)')

# Show last session status immediately after pull
try:
    from agent.tools.check_last_session import check_last_session
    print('\n── Last session status ──')
    check_last_session()
except Exception as _e:
    print(f'  (no session history yet: {_e})')

print('\n' + '='*60)
print('  BOOTSTRAP COMPLETE')
print(f'  Runtime : {RUNTIME.upper()}')
print(f'  Root    : {PROJECT_ROOT}')
print('  --> Run Cell 2 to build / verify the data pipeline.')
print('='*60)

In [ ]:
# [OLLAMA] Start Ollama server if not running
import subprocess, time, os

OLLAMA_BIN = '/home/dhruval/ollama'

# Check if Ollama already running
result = subprocess.run(
    ['curl', '-s', 'http://localhost:11434/api/tags'],
    capture_output=True
)

if result.returncode != 0:
    print('Ollama not running — starting...')
    subprocess.Popen(
        [OLLAMA_BIN, 'serve'],
        stdout=open('/home/dhruval/ollama.log', 'w'),
        stderr=subprocess.STDOUT,
        env={**os.environ,
             'OLLAMA_MODELS': '/home/dhruval/.ollama/models',
             'OLLAMA_HOST': '127.0.0.1:11434'}
    )
    time.sleep(5)
    r = subprocess.run(
        ['curl', '-s', 'http://localhost:11434/api/tags'],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print('Ollama started successfully')
    else:
        print('WARNING: Ollama failed to start')
        print(open('/home/dhruval/ollama.log').read())
else:
    print('Ollama already running')

# Show available models
r = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
print('\nDownloaded models:')
print(r.stdout or '  (none yet)')

In [ ]:
# [OLLAMA] Verify GPU usage
import subprocess

OLLAMA_BIN = '/home/dhruval/ollama'

r = subprocess.run([OLLAMA_BIN, 'ps'], capture_output=True, text=True)
print('Models currently in VRAM:')
print(r.stdout or '  (none loaded yet — will load on first agent call)')

r2 = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print(f'\nGPU status: {r2.stdout.strip()}')

In [ ]:
# ── Path guard: works even if Cell 1 hasn't run (import agent installed via pip install -e .) ──
import sys, os; _r = next((p for p in [
    __import__('pathlib').Path(r'd:\3rd sem M.tech\agent_workspace'),
    __import__('pathlib').Path('/home/dhruval/agent_workspace'),
    __import__('pathlib').Path('/mnt/d/3rd sem M.tech/agent_workspace'),
    __import__('pathlib').Path.home()/'agent_workspace'
] if p.exists()), None)
if _r and str(_r) not in sys.path: sys.path.insert(0, str(_r))
if _r: os.chdir(_r)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 2 · [VERIFY DATA]  Load CSV -> features -> reduce -> splits -> save  ║
# ║  Safe to re-run: skips expensive feature engineering if splits exist.      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import numpy as np
from pathlib import Path
from agent.data.pipeline import DataPipeline
from agent.core.tee_logger import TeeLogger

logger = TeeLogger(master_log_dir='master_log')
logger.info('=== Phase 1: Data Verification Start ===')

pipe = DataPipeline(splits_dir='data/splits', random_state=42)

_train_pkl = Path('data/splits/train.pkl')
_selector  = Path('data/processed/selector.pkl')
_weights   = Path('data/processed/class_weights.pkl')

if _train_pkl.exists() and _selector.exists() and _weights.exists():
    logger.info('Existing pipeline artifacts found — loading.')
    pipe.load_splits()
else:
    csv_path = Path('data/raw/alpha_synuclein.csv')
    if not csv_path.exists():
        raise FileNotFoundError(
            f'CSV not found: {csv_path}\n'
            'It is committed to GitHub — check that the clone succeeded.'
        )
    logger.info('Loading and expanding CSV ...')
    df = pipe.load_and_expand(str(csv_path))
    logger.info(f'Expanded DataFrame: {df.shape}')

    logger.info('Engineering features (2-mer + 3-mer) ...')
    X, y = pipe.build_features(df)
    logger.info(f'Raw feature matrix: {X.shape}')

    logger.info('Stratified split 70/15/15 ...')
    pipe.stratified_split(X, y, train=0.70, val=0.15, test=0.15)

    logger.info('Reducing features (VarianceThreshold + SelectKBest k=500) ...')
    pipe.reduce_features(
        pipe.X_train, pipe.X_val, pipe.X_test, pipe.y_train, k_best=500
    )

    logger.info('Computing class weights ...')
    pipe.get_class_weights(pipe.y_train)

    logger.info('Saving splits ...')
    pipe.save_splits()

print()
label_names = {0: 'No', 1: 'Low', 2: 'Medium', 3: 'High'}
unique, counts = np.unique(pipe.y_train, return_counts=True)
print('='*52)
print(f'Train : {pipe.X_train.shape}  Val: {pipe.X_val.shape}  Test: {pipe.X_test.shape}')
for u, c in zip(unique.tolist(), counts.tolist()):
    print(f'  Class {u} ({label_names[u]:<6}): {c} train samples')
print('data/processed/ artifacts:')
for f in sorted(Path('data/processed').glob('*.pkl')):
    print(f'  {f.name}  ({f.stat().st_size:,} B)')
print('='*52)
logger.info('=== Phase 1: Data Verification Complete ===')

In [ ]:
# ── Path guard: works even if Cell 1 hasn't run (import agent installed via pip install -e .) ──
import sys, os; _r = next((p for p in [
    __import__('pathlib').Path(r'd:\3rd sem M.tech\agent_workspace'),
    __import__('pathlib').Path('/home/dhruval/agent_workspace'),
    __import__('pathlib').Path('/mnt/d/3rd sem M.tech/agent_workspace'),
    __import__('pathlib').Path.home()/'agent_workspace'
] if p.exists()), None)
if _r and str(_r) not in sys.path: sys.path.insert(0, str(_r))
if _r: os.chdir(_r)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 3 · [VERIFY WALL]  Seal and verify the mathematical wall             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
from agent.core.tee_logger import TeeLogger

logger = TeeLogger(master_log_dir='master_log')

logger.info('Sealing test set ...')
digest = pipe.seal_test_set()

logger.info('Verifying mathematical wall integrity ...')
wall_intact = pipe.verify_wall()

print()
print('='*60)
print(f'  SHA-256 : {digest}')
print(f'  Wall    : {"INTACT" if wall_intact else "*** BREACH DETECTED ***"}')
print('='*60)
print()
print('CROSS-MACHINE NOTE:')
print('  Copy the SHA-256 above and compare it to the hash')
print('  produced by running this cell on the cloud instance.')
print('  Matching = reproducible test sets. Different = invalid comparisons.')

if wall_intact:
    logger.info('Mathematical wall INTACT. Safe to proceed.')
else:
    logger.error('WALL BREACH — test set has been tampered with!')
    raise RuntimeError('Wall breach detected. Do not run experiments.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 3b · [DEV RESET]  Wipe experiments + logs for a fresh run           ║
# ║  NEVER deletes data/, agent/, .env, or .git/  — mathematical wall safe.   ║
# ║  Skip this cell for production — only use during development.             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import sys, os
_r = next((p for p in [
    __import__('pathlib').Path(r'd:\3rd sem M.tech\agent_workspace'),
    __import__('pathlib').Path('/home/dhruval/agent_workspace'),
    __import__('pathlib').Path('/mnt/d/3rd sem M.tech/agent_workspace'),
    __import__('pathlib').Path.home()/'agent_workspace'
] if p.exists()), None)
if _r and str(_r) not in sys.path: sys.path.insert(0, str(_r))
if _r: os.chdir(_r)

from agent.tools.dev_reset import dev_reset

# ── Safety Input (Plain Text) ──────────────────────────────────────────────────
# This blocks 'Run All' from deleting data without manual intervention
print("DEV RESET: This will wipe experiments and logs.")
confirmation = input("Type 'RESET' to confirm (or anything else to skip): ")

if confirmation.strip().upper() == 'RESET':
    print("\nExecuting DEV RESET...")
    dev_reset()
    print("Done. Experiments and logs cleared.")
else:
    print("\nReset skipped. No changes made.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 4b · [OLLAMA]  Start Ollama if not running (IIT GN server only)      ║
# ║  Safe to run on other runtimes — no-op unless RUNTIME == 'iit'            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import subprocess
import time
import os

OLLAMA_BIN = '/home/dhruval/ollama'

# RUNTIME is set by Cell 1. If this cell is run standalone, default to 'local'.
_runtime = globals().get('RUNTIME', 'local')

if _runtime == 'iit':
    print('IIT GN server detected — checking Ollama status...')
    
    # Check if a server instance is already running on port 11434
    result = subprocess.run(
        ['curl', '-s', 'http://127.0.0.1:11434/api/tags'],
        capture_output=True
    )
    
    if result.returncode != 0:
        print('Ollama not running — starting ollama serve in background...')
        
        # Build strict user-space environment variables ensuring system path inheritance
        user_env = os.environ.copy()
        user_env['OLLAMA_MODELS'] = '/home/dhruval/.ollama/models'
        user_env['OLLAMA_HOST'] = '127.0.0.1:11434'
        
        # Start server process safely redirecting logs
        with open('/home/dhruval/ollama.log', 'w') as log_file:
            subprocess.Popen(
                [OLLAMA_BIN, 'serve'],
                stdout=log_file,
                stderr=subprocess.STDOUT,
                env=user_env
            )
            
        # Give the server 5 seconds to wake up and map to the Ada GPU
        time.sleep(5)
        
        check = subprocess.run(
            ['curl', '-s', 'http://127.0.0.1:11434/api/tags'],
            capture_output=True
        )
        
        if check.returncode == 0:
            print('Ollama server started successfully.')
            print('Log location: /home/dhruval/ollama.log')
        else:
            print('WARNING: Ollama failed to bind — printing latest log output:')
            with open('/home/dhruval/ollama.log', 'r') as f:
                print(f.read())
            print('\nAlternative: Run manually in terminal: /home/dhruval/ollama serve &')
    else:
        print('Ollama already running cleanly at http://127.0.0.1:11434')

    # Explicitly pull model list using user environment settings
    r = subprocess.run(
        [OLLAMA_BIN, 'list'], 
        env={'OLLAMA_MODELS': '/home/dhruval/.ollama/models'}, 
        capture_output=True, 
        text=True
    )
    print('\nDownloaded models inside user space:')
    print(r.stdout.strip() or '  (none yet)')

    # Display GPU Allocation
    r2 = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'\nGPU status: {r2.stdout.strip()}')

else:
    print(f'Runtime is {_runtime!r} — Ollama auto-start skipped.')
    print("(Ollama auto-start only runs on RUNTIME='iit')")

In [ ]:
# ── Path guard ──────────────────────────────────────────────────────────────────────────
import sys, os
_r = next((p for p in [
    __import__('pathlib').Path(r'd:\3rd sem M.tech\agent_workspace'),
    __import__('pathlib').Path('/home/dhruval/agent_workspace'),
    __import__('pathlib').Path('/mnt/d/3rd sem M.tech/agent_workspace'),
    __import__('pathlib').Path.home()/'agent_workspace'
] if p.exists()), None)
if _r and str(_r) not in sys.path: sys.path.insert(0, str(_r))
if _r: os.chdir(_r)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 4 · [LAUNCH AGENT]  Start autonomous ML research loop               ║
# ║  Concise mode: one line per step in cell; full log → session_log.log      ║
# ║  Stop button: graceful stop — current experiment finishes first           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
from dotenv import load_dotenv
load_dotenv()

import ipywidgets as widgets
from IPython.display import display
from agent.core.orchestrator import AgentOrchestrator

# ── Configuration — edit these before launching ─────────────────────────────────
MODEL_NAME        = 'local-qwen3-coder'   # local-qwen, local-qwen3-coder, local-qwen3, groq-llama, gemini-flash, gemini-flash-lite...
MAX_EXPERIMENTS   = 100            # soft cap injected into the prompt
VERBOSITY         = 'full'         # 'concise' (recommended) | 'full'
MAX_IDLE_SECONDS  = 600            # watchdog: stop if stuck for 5 min
MAX_TOTAL_SECONDS = 7200           # watchdog: stop after 2 h total

# ── Build agent ───────────────────────────────────────────────────────────────────
agent = AgentOrchestrator(
    model_name        = MODEL_NAME,
    verbosity         = VERBOSITY,
    max_idle_seconds  = MAX_IDLE_SECONDS,
    max_total_seconds = MAX_TOTAL_SECONDS,
)
print('Agent status:', agent.status())

# ── Stop button (ipywidgets) ────────────────────────────────────────────────────────
_stop_btn = widgets.Button(
    description='⏹  Stop Agent',
    button_style='warning',
    layout=widgets.Layout(width='160px', height='36px'),
    tooltip='Graceful stop — current experiment finishes first',
)
_status_lbl = widgets.Label('Status: idle')

def _on_stop(_btn):
    _stop_btn.disabled = True
    _status_lbl.value = 'Status: stop requested — waiting for current step to finish ...'
    agent.stop()

_stop_btn.on_click(_on_stop)
display(widgets.HBox([_stop_btn, _status_lbl]))

# ── Launch ──────────────────────────────────────────────────────────────────────────
print(f'\nStarting autonomous research loop (max {MAX_EXPERIMENTS} experiments) ...')
print('Click "Stop Agent" above for a graceful stop.')
print('Interrupt kernel (■) for immediate stop (may leave experiment mid-run).\n')

_status_lbl.value = 'Status: running'
try:
    result = agent.run(
        max_experiments   = MAX_EXPERIMENTS,
        max_idle_seconds  = MAX_IDLE_SECONDS,
        max_total_seconds = MAX_TOTAL_SECONDS,
    )
    if result:
        print('\nAgent final answer:', result)
finally:
    _stop_btn.disabled = True
    _status_lbl.value  = 'Status: finished'

In [ ]:
# ── Path guard: works even if Cell 1 hasn't run (import agent installed via pip install -e .) ──
import sys, os; _r = next((p for p in [
    __import__('pathlib').Path(r'd:\3rd sem M.tech\agent_workspace'),
    __import__('pathlib').Path('/home/dhruval/agent_workspace'),
    __import__('pathlib').Path('/mnt/d/3rd sem M.tech/agent_workspace'),
    __import__('pathlib').Path.home()/'agent_workspace'
] if p.exists()), None)
if _r and str(_r) not in sys.path: sys.path.insert(0, str(_r))
if _r: os.chdir(_r)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 5 · [DASHBOARD]  Live combined leaderboard — both machines          ║
# ║  Rebuilds leaderboard from disk each refresh (merges both machines).      ║
# ║  Shows current session status at the top.                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import json, time, subprocess, threading
from pathlib import Path
from IPython.display import clear_output, display, HTML
import ipywidgets as widgets

from agent.tools.rebuild_leaderboard import rebuild_leaderboard
from agent.tools.check_last_session import get_last_session_info, _ago, _status_icon

_STATE_PATH = Path('master_log/orchestrator_state.json')
_REFRESH_S  = 30

def _gpu_info():
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=3
        )
        if r.returncode == 0:
            used, total, util = r.stdout.strip().split(',')
            return f'GPU {used.strip()}/{total.strip()} MB | {util.strip()}% util'
    except Exception:
        pass
    return 'GPU: n/a'

def _read_state():
    try:
        return json.loads(_STATE_PATH.read_text()) if _STATE_PATH.exists() else {}
    except Exception:
        return {}

def _bar(done, total=200, w=40):
    f = int(w * done / max(total, 1))
    return f'[{"#"*f + "-"*(w-f)}] {done}/{total} ({100*done/max(total,1):.1f}%)'

def _session_html():
    """Render current session info as an HTML block."""
    info = get_last_session_info()
    if not info:
        return '<p style="color:#78909c">No session history found.</p>'

    icon   = _status_icon(info['status'])
    sc     = {'completed':'#66bb6a','running':'#4fc3f7','interrupted':'#ffa726',
               'crashed':'#ef5350','starting':'#b0bec5'}.get(info['status'],'#78909c')
    hb_ago = _ago(info['hb_time'])
    stale  = info['status'] == 'running' and 'hour' in hb_ago
    stale_warn = (
        '<span style="color:#ef5350"> ⚠ heartbeat stale — may have crashed</span>'
        if stale else ''
    )
    err_html = ''
    if info.get('error'):
        err_html = f'<br><b style="color:#ef5350">Error:</b> {info["error"][:150]}'

    return (
        f'<div style="background:#131c22;border-left:3px solid {sc};padding:8px 14px;'
        f'border-radius:4px;margin-bottom:8px;font-size:.85em">'
        f'<b style="color:{sc}">{icon} Session: {info["session_id"]}</b>'
        f' &nbsp;|&nbsp; <span style="color:#90caf9">{info["model"]}</span>'
        f' &nbsp;|&nbsp; machine: <span style="color:#80cbc4">{info["machine"]}</span>'
        f'<br>Status: <span style="color:{sc}">{(info["status"] or "?").upper()}</span>'
        f'&nbsp;&nbsp;Experiments: <b>{info["experiments"]}</b> this session'
        f'&nbsp;&nbsp;Duration: {info["duration"]}'
        f'<br>Last exp: <span style="color:#b0bec5">{info["hb_exp"]}</span>'
        f'&nbsp;&nbsp;Heartbeat: <span style="color:#78909c">{hb_ago}</span>'
        f'{stale_warn}{err_html}'
        f'</div>'
    )

def _render(out):
    try:
        lb = rebuild_leaderboard(verbose=False)
    except Exception as exc:
        lb = {}
        print(f'rebuild_leaderboard error: {exc}')

    state = _read_state()
    exps  = lb.get('experiments', [])
    n     = lb.get('total_runs', 0)
    f1    = lb.get('best_val_f1_macro', 0.0)
    best  = lb.get('best_experiment', 'none')
    fams  = lb.get('families_completed', [])
    upd   = lb.get('last_updated', 'never')
    mach  = lb.get('runs_per_machine', {})
    st    = state.get('status', 'unknown')
    model = state.get('model', '?')
    sc    = {'running':'#66bb6a','idle':'#bdbdbd','stopping':'#ffa726'}.get(st,'#ef5350')
    mach_str = '  '.join(f'{m}:{c}' for m, c in sorted(mach.items()))

    rows = exps[:10]
    tbl  = '<table style="border-collapse:collapse;width:100%;font-size:.84em">'
    tbl += '<tr style="background:#1e2a32">'
    for h in ['#','ID','Architecture','Family','Machine','F1-macro','Acc','Time','Status']:
        tbl += f'<th style="padding:4px 8px;color:#90caf9;text-align:left">{h}</th>'
    tbl += '</tr>'
    for i, e in enumerate(rows, 1):
        v  = e.get('val_f1_macro',0)
        vc = '#66bb6a' if v>=.6 else ('#ffa726' if v>=.3 else '#ef5350')
        sc2= '#66bb6a' if e.get('status')=='success' else '#ef5350'
        bg = '#131c22' if i%2 else '#0d1117'
        mid = e.get('machine_id','?')
        mc  = '#80cbc4' if mid == 'gcloud' else '#ffcc80'
        tbl += (
            f'<tr style="background:{bg}">'
            f'<td style="padding:3px 8px;color:#78909c">{i}</td>'
            f'<td style="padding:3px 8px;color:#b0bec5;font-size:.78em">{e.get("exp_id","?")[:35]}</td>'
            f'<td style="padding:3px 8px;color:#e0e0e0">{e.get("architecture","?")[:20]}</td>'
            f'<td style="padding:3px 8px;color:#80cbc4">{e.get("architecture_family","?")[:13]}</td>'
            f'<td style="padding:3px 8px;color:{mc};font-size:.78em">{mid}</td>'
            f'<td style="padding:3px 8px;font-weight:bold;color:{vc}">{v:.4f}</td>'
            f'<td style="padding:3px 8px;color:#78909c">{e.get("val_accuracy",0):.4f}</td>'
            f'<td style="padding:3px 8px;color:#78909c">{e.get("train_time_seconds",0):.1f}s</td>'
            f'<td style="padding:3px 8px;color:{sc2}">{e.get("status","?")}</td>'
            f'</tr>'
        )
    tbl += '</table>'

    ALL = ['classical_ml','linear','neural_network','deep_residual',
           'ensemble_stack','attention_based','graph_neural','automl']
    pills = ''.join(
        f'<span style="background:{"#2e7d32" if fa in fams else "#37474f"};'
        f'color:#e0e0e0;border-radius:4px;padding:2px 8px;margin:2px;'
        f'font-size:.8em;display:inline-block">{"✓" if fa in fams else "·"} {fa}</span>'
        for fa in ALL
    )

    html = (
        f'<div style="background:#0d1117;color:#e0e0e0;padding:16px;border-radius:8px;font-family:monospace">'
        f'<h2 style="margin:0 0 4px;color:#4fc3f7">Alpha-Synuclein Agent — Combined Leaderboard</h2>'
        f'<small>Rebuilt from disk each refresh | Updated: {upd}</small><br><br>'
        + _session_html() +
        f'<b>Agent:</b> <span style="color:{sc}">{st.upper()}</span>&nbsp;&nbsp;'
        f'<b>Model:</b> {model}&nbsp;&nbsp;<b>{_gpu_info()}</b><br>'
        f'<b>Progress:</b> {_bar(n)}&nbsp;&nbsp;'
        f'<b>Machines:</b> <span style="color:#80cbc4">{mach_str or "none yet"}</span><br>'
        f'<b>Best F1-macro:</b> <span style="font-size:1.3em;color:#ffb74d">{f1:.4f}</span>'
        f' ({best})<br><br>'
        f'{tbl}<br>'
        f'<b>Families:</b><br>{pills}'
        f'</div>'
    )
    with out:
        clear_output(wait=True)
        display(HTML(html))

out      = widgets.Output()
_stop    = [False]
btn_stop = widgets.Button(description='Stop', button_style='danger', icon='stop')
btn_now  = widgets.Button(description='Refresh', button_style='info', icon='refresh')

btn_stop.on_click(lambda _: [_stop.__setitem__(0, True),
                              btn_stop.__setattr__('disabled', True)])
btn_now.on_click(lambda _: _render(out))

display(widgets.HBox([btn_now, btn_stop]))
display(out)
_render(out)

def _loop():
    while not _stop[0]:
        for _ in range(_REFRESH_S * 2):
            if _stop[0]: return
            time.sleep(0.5)
        _render(out)

threading.Thread(target=_loop, daemon=True).start()
print(f'Dashboard running — rebuilt from disk every {_REFRESH_S}s.')

In [ ]:
import os

folder_path = "experiments"

# Get only directories
folders = [
    name for name in os.listdir(folder_path)
    if os.path.isdir(os.path.join(folder_path, name))
]

for folder in folders:
    print(folder)

In [ ]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 4c · [OLLAMA]  Stop Ollama Server and Free GPU Memory                ║
# ║  Safe to run on other runtimes — no-op unless RUNTIME == 'iit'            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import subprocess, time

_runtime = globals().get('RUNTIME', 'local')

if _runtime == 'iit':
    print('IIT GN server detected — stopping Ollama server...')
    
    # Forcefully terminate any running ollama processes
    subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
    time.sleep(2)  # Give the card a moment to flush cache
    
    # Check if port is still active
    check = subprocess.run(
        ['curl', '-s', 'http://127.0.0.1:11434/api/tags'],
        capture_output=True
    )
    
    if check.returncode != 0:
        print('Ollama server stopped successfully. Port 11434 released.')
    else:
        print('WARNING: Process might still be closing. Retrying kill...')
        subprocess.run(['pkill', '-9', '-f', 'ollama'])

    # Show updated GPU status to verify VRAM is completely free
    r2 = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'\nUpdated GPU status: {r2.stdout.strip()}')

else:
    print(f'Runtime is {_runtime!r} — Stop script skipped.')